# Maintenance Rehearsal (A) — Training (extractive)

Trains the `roberta-base` sentence scorer from
`src/pipeline/extractive.py` on the per-sentence labels built in
`04_rehearsal_maintenance_prep.ipynb`.

A plain PyTorch loop rather than `Seq2SeqTrainer`: this is binary
classification over a ragged number of sentences per example, which does not
fit the seq2seq Trainer's assumptions, and the loop is short enough to read.

## What to judge this model on

The novel n-gram check that mattered for the seq2seq version changes shape
here, and the obvious version of it is wrong. An extractive model only emits
source sentences, but **joining non-adjacent ones creates n-grams that span
the gap** — the tail of one kept sentence followed by the head of the next.
So the ratio is *not* 0, and it grows with the compression rate.

The invariant that does hold is per-sentence: every kept sentence must appear
verbatim in the source, so **every novel n-gram must span a seam**. One
sitting inside a single kept sentence means that sentence was altered, which
is a real defect. `seam_report` (§6) checks exactly that. The "is it
abstractive enough" side of the manipulation check has moved to `07`, where B
has to prove it is *not* extractive.

What replaces it:

1. **Sentence-level F1** against the oracle labels — did it learn which
   sentences carry the summary.
2. **Selection ROUGE-L**, comparing the sentences the model picks against the
   ones the oracle picked, at the ratio actually used at inference. This is
   closer to the deployed behaviour than raw classification accuracy, because
   at inference we take the top-k by score rather than thresholding.
3. **Positive rate sanity** — with roughly 2-3 positives among ~10 sentences,
   a model that predicts all-zero scores a deceptively good accuracy. F1 and
   the selection metric are the ones that catch that.

In [3]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()
API_KEY_SET = bool(__import__("os").environ.get("NVIDIA_NIM_API_KEY"))

import json
import math

import datasets
import evaluate
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

from src.pipeline.extractive import (
    SentenceScorer,
    collate,
    score_sentences,
    seam_report,
    select_sentences,
)

ModuleNotFoundError: No module named 'src'

## 1. Load data and model

In [ ]:
MODEL_NAME = "roberta-base"
DATA_DIR = Path("data/processed/rehearsal_maintenance")
OUTPUT_DIR = Path("experiments/rehearsal_maintenance_roberta")
BATCH_SIZE = 8
EPOCHS = 8
LEARNING_RATE = 2e-5

DATA_READY = all((DATA_DIR / split).exists() for split in ("train", "val", "test"))
if not DATA_READY:
    print(f"No data at {DATA_DIR} — run 04_rehearsal_maintenance_prep.ipynb first.")
else:
    train_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "train"))
    val_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "val"))
    test_dataset = datasets.Dataset.load_from_disk(str(DATA_DIR / "test"))
    print(f"train {len(train_dataset)} / val {len(val_dataset)} / test {len(test_dataset)}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = SentenceScorer(MODEL_NAME)
    device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"{MODEL_NAME} on {device}, parameters: {sum(p.numel() for p in model.parameters()):,}")

    def make_loader(ds, shuffle):
        return DataLoader(
            ds, batch_size=BATCH_SIZE, shuffle=shuffle,
            collate_fn=lambda b: collate(b, tokenizer.pad_token_id),
        )

    train_loader = make_loader(train_dataset, True)
    val_loader = make_loader(val_dataset, False)
    test_loader = make_loader(test_dataset, False)

## 2. Metrics

`evaluate` runs over whole batches, masking out padded sentence slots. The
all-zero baseline is printed alongside: with ~25% positives, predicting
nothing scores ~75% accuracy, so accuracy alone would look fine for a model
that has learned nothing.

In [ ]:
def evaluate_loader(model, loader) -> dict:
    model.eval()
    tp = fp = fn = tn = 0
    losses = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            losses.append(out["loss"].item())

            mask = batch["cls_mask"].bool()
            pred = (torch.sigmoid(out["logits"]) > 0.5)[mask]
            gold = batch["labels"].bool()[mask]
            tp += (pred & gold).sum().item()
            fp += (pred & ~gold).sum().item()
            fn += (~pred & gold).sum().item()
            tn += (~pred & ~gold).sum().item()

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    total = tp + fp + fn + tn
    return {
        "loss": float(np.mean(losses)),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": (tp + tn) / total if total else 0.0,
        "all_zero_accuracy": (tn + fp) / total if total else 0.0,
        "positive_rate": (tp + fn) / total if total else 0.0,
    }

## 3. Train

Linear warmup then decay, the standard BERT-family fine-tuning schedule. The
best epoch by validation F1 is kept — not by loss, because loss is dominated
by the majority-negative class.

In [ ]:
if DATA_READY:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    best_f1, history = -1.0, []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = []
        for batch in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = model(**batch)["loss"]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            running.append(loss.item())

        metrics = evaluate_loader(model, val_loader)
        history.append({"epoch": epoch, "train_loss": float(np.mean(running)), **metrics})
        print(f"  epoch {epoch}: train_loss {np.mean(running):.4f} | "
              f"val f1 {metrics['f1']:.4f} P {metrics['precision']:.3f} R {metrics['recall']:.3f} "
              f"| acc {metrics['accuracy']:.3f} (all-zero baseline {metrics['all_zero_accuracy']:.3f})")

        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            torch.save(model.state_dict(), OUTPUT_DIR / "sentence_scorer.pt")
            print(f"    new best (val f1 {best_f1:.4f}) — saved")

    import pandas as pd

    print()
    print(pd.DataFrame(history).round(4).to_string(index=False))

## 4. Save

In [ ]:
if DATA_READY:
    model.load_state_dict(torch.load(OUTPUT_DIR / "sentence_scorer.pt"))
    tokenizer.save_pretrained(OUTPUT_DIR)
    (OUTPUT_DIR / "config.json").write_text(
        json.dumps({"model_name": MODEL_NAME, "epochs": EPOCHS, "lr": LEARNING_RATE,
                    "batch_size": BATCH_SIZE, "best_val_f1": best_f1}, indent=2),
        encoding="utf-8",
    )
    print(f"Saved to: {OUTPUT_DIR}  (best val f1 {best_f1:.4f})")

## 5. Final evaluation — held-out test set

Never used for gradient updates or checkpoint selection. Two views: the
classification metrics, and the selection metric that matches how the model
is actually used (top-k by score at a given ratio, not a 0.5 threshold).

In [ ]:
if DATA_READY:
    test_metrics = evaluate_loader(model, test_loader)
    print("classification:", {k: round(v, 4) for k, v in test_metrics.items()})

    rouge = evaluate.load("rouge")
    raw = pd.read_csv(DATA_DIR / "test_sentences_raw.csv", encoding="utf-8-sig")

    RATIO = 0.3  # for reporting only; at inference this comes from dependent_ratio()
    preds, refs = [], []
    for _, row in raw.head(200).iterrows():
        sentences = json.loads(row["sentences"])
        labels = json.loads(row["labels"])
        oracle = [s for s, keep in zip(sentences, labels) if keep]
        if not oracle:
            continue
        scores = score_sentences(sentences, model, tokenizer)
        preds.append(" ".join(select_sentences(sentences, scores, RATIO)))
        refs.append(" ".join(oracle))

    result = rouge.compute(predictions=preds, references=refs, use_stemmer=False)
    print(f"\nselection ROUGE-L vs oracle sentences (ratio={RATIO}, n={len(preds)}): "
          f"{result['rougeL']:.4f}")
    print(f"selection ROUGE-1: {result['rouge1']:.4f}")

## 6. Qualitative check on a real chunked article

Runs the trained scorer through the actual pipeline path — chunk a held-out
`cnn_dailymail` test article, then compress each chunk with the dependent
ratio — and prints what survives.

`target_context_tokens` is the downstream budget K; the ratio R = K/|D| falls
out of it, which is the whole point of the dependent-ratio strategy (Sie et
al. §3.2). Compression here is a **single pass**: the ratio is sized so one
pass already fits.

In [ ]:
if not DATA_READY:
    print("No data — skipping.")
elif not API_KEY_SET:
    print("No NVIDIA_NIM_API_KEY (chunking needs the embedding API) — skipping.")
else:
    import yaml
    from datasets import load_dataset

    from src.pipeline.chuncking import paginate_semantic, plain_text_to_paragraphs
    from src.pipeline.embeddings import embed_texts, load_config
    from src.pipeline.gisting import split_into_sentences
    from src.pipeline.rehearsal import dependent_ratio

    article = load_dataset("cnn_dailymail", "3.0.0", split="test[0:1]")[0]["article"]
    chunk_cfg = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))
    embed_cfg = load_config("configs/importance_filter.yaml")
    chunks = paginate_semantic(
        plain_text_to_paragraphs(article),
        min_words=chunk_cfg["min_words"], max_words=chunk_cfg["max_words"],
        granularity="paragraph", config=embed_cfg, embed_fn=embed_texts,
    )

    # Small on purpose: this demo article is only ~5 chunks, so a generous K
    # would give R near 1 and compress nothing visible.
    TARGET_CONTEXT_TOKENS = 150
    ratio = dependent_ratio(chunks, tokenizer, TARGET_CONTEXT_TOKENS)
    print(f"{len(chunks)} chunks | K={TARGET_CONTEXT_TOKENS} -> dependent ratio R={ratio:.3f}\n")

    all_ok = True
    for chunk in chunks:
        sentences = [s for s in split_into_sentences(chunk.text) if s.strip()]
        scores = score_sentences(sentences, model, tokenizer)
        kept = select_sentences(sentences, scores, ratio)
        compressed = " ".join(kept)
        report = seam_report(kept, chunk.text, n=3)
        all_ok &= report["ok"]

        print(f"[chunk {chunk.index}] {len(sentences)} sentences -> {len(kept)} | "
              f"{len(chunk.text)} -> {len(compressed)} chars "
              f"({len(compressed) / max(1, len(chunk.text)):.1%})")
        print(f"    verbatim {'OK' if report['ok'] else 'FAILED'} | "
              f"novel 3-grams {report['novel_total']} across {report['seams']} seams "
              f"(ratio {report['novel_ratio']:.4f}) | "
              f"altered-sentence violations {len(report['novel_inside_sentence'])}")
        if report["not_verbatim"]:
            print(f"    *** sentences not found in source: {report['not_verbatim'][:2]}")
        print(f"    {compressed[:150]}")

    print()
    print("seam check:", "PASS — every novel n-gram spans a seam" if all_ok
          else "*** FAIL — a kept sentence was altered")

## Summary

_To be filled in after the run._

Read in this order:

1. **val/test F1 against the all-zero baseline.** With ~25% positives, an
   untrained-looking model still scores ~75% accuracy. If F1 is near zero
   while accuracy looks respectable, the model has learned to predict nothing.
2. **Selection ROUGE-L** (§5) — closer to deployed behaviour than the 0.5
   threshold, because inference takes top-k by score.
3. **`seam_report` must report `ok=True` on every chunk** (§6). A non-zero
   novel n-gram count is expected and grows with compression — what must be
   zero is `novel_inside_sentence`, i.e. altered sentences.

Next: wire this scorer into the pipeline in place of the seq2seq path in
`src/pipeline/rehearsal.py`, which stays in the repo for now so the earlier
results remain reproducible.